# Internal cohort and patient-level splits

**Scientific purpose.** Construct the PLC-CECT V4 three-class cohort and the
patient-level development, held-out internal evaluation, and five-fold assignments
used by the reported study.

**Runnability classification:** runnable from the user-obtained PLC-CECT V4 release
and configured private storage. The original patient-level split files are intentionally
not distributed; executing this notebook creates patient-linked files only under the
configured private artifact root.

**Inputs:** `patient_data.csv` from PLC-CECT V4, referenced through
`paths.plc_cect_root` in the untracked local configuration.

**Private outputs:** `train_val_test.json` and `fold_0.json` through `fold_4.json`.

**Expected aggregate invariants:** 278 patients (94 HCC, 99 ICC, 85 cHCC-CCA),
222 development patients, 56 held-out patients, five patient-level stratified folds.
All phases belonging to a patient remain in the same partition.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import json
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

INTERNAL_ROOT = require_path(CONFIG, "plc_cect_root")
MANIFEST_PATH = INTERNAL_ROOT / "patient_data.csv"
SPLIT_ROOT = PRIVATE_ARTIFACT_ROOT / "splits"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "PLC-CECT patient_data.csv is unavailable at the configured dataset root."
    )


## Cohort construction

The source token `CHCC` is retained in private serialized artifacts for compatibility;
its public display label is `cHCC-CCA`. The code checks that each patient has a single
class label before splitting and displays aggregate counts only.


In [ ]:
ARTIFACT_CLASS_ORDER = ("HCC", "ICC", "CHCC")
EXPECTED_COUNTS = {"HCC": 94, "ICC": 99, "CHCC": 85}

raw = pd.read_csv(MANIFEST_PATH)
required_columns = {"patient_id", "cancer_type"}
missing_columns = sorted(required_columns.difference(raw.columns))
if missing_columns:
    raise ValueError(f"Internal manifest is missing required columns: {missing_columns}")

label_aliases = {"HCC": "HCC", "ICC": "ICC", "CHCC": "CHCC", "cHCC-CCA": "CHCC"}
cohort_rows = raw.loc[:, ["patient_id", "cancer_type"]].copy()
cohort_rows["cancer_type"] = cohort_rows["cancer_type"].map(label_aliases)
cohort_rows = cohort_rows[cohort_rows["cancer_type"].isin(ARTIFACT_CLASS_ORDER)]

label_counts_per_patient = cohort_rows.groupby("patient_id")["cancer_type"].nunique()
if not (label_counts_per_patient == 1).all():
    raise ValueError("At least one patient has conflicting class labels in the source manifest.")

patients = cohort_rows.drop_duplicates("patient_id").reset_index(drop=True)
observed_counts = patients["cancer_type"].value_counts().to_dict()
if observed_counts != EXPECTED_COUNTS:
    raise ValueError(
        f"PLC-CECT cohort counts differ from the locked study cohort: {observed_counts}"
    )
pd.DataFrame(
    {"class": ["HCC", "ICC", "cHCC-CCA"], "patients": [94, 99, 85]}
)


## Locked split procedure

The historical random seed was 42. The held-out fraction was 0.20, followed by a
shuffled five-fold stratified split of the 222-patient development cohort.


In [ ]:
SEED = 42
patient_ids = patients["patient_id"].astype(str).to_numpy()
labels = patients["cancer_type"].to_numpy()

dev_ids, test_ids = train_test_split(
    patient_ids,
    test_size=0.20,
    stratify=labels,
    random_state=SEED,
)
dev_ids = sorted(dev_ids.tolist())
test_ids = sorted(test_ids.tolist())
if len(dev_ids) != 222 or len(test_ids) != 56 or set(dev_ids).intersection(test_ids):
    raise RuntimeError("Patient-level development/evaluation split invariant failed.")

development = patients[patients["patient_id"].astype(str).isin(dev_ids)].copy()
development = development.reset_index(drop=True)
folds = []
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold_index, (train_index, validation_index) in enumerate(
    splitter.split(development["patient_id"], development["cancer_type"])
):
    train_fold_ids = sorted(development.iloc[train_index]["patient_id"].astype(str).tolist())
    validation_fold_ids = sorted(
        development.iloc[validation_index]["patient_id"].astype(str).tolist()
    )
    if set(train_fold_ids).intersection(validation_fold_ids):
        raise RuntimeError(f"Fold {fold_index} has patient overlap.")
    folds.append(
        {
            "fold": fold_index,
            "train_ids": train_fold_ids,
            "val_ids": validation_fold_ids,
        }
    )

validation_coverage = [pid for fold in folds for pid in fold["val_ids"]]
if sorted(validation_coverage) != sorted(dev_ids):
    raise RuntimeError("Each development patient must occur in exactly one validation fold.")


In [ ]:
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
main_split = {
    "seed": SEED,
    "classes": list(ARTIFACT_CLASS_ORDER),
    "test_frac": 0.20,
    "n_folds": 5,
    "dev_ids": dev_ids,
    "test_ids": test_ids,
}
(SPLIT_ROOT / "train_val_test.json").write_text(
    json.dumps(main_split, indent=2), encoding="utf-8"
)
for fold in folds:
    (SPLIT_ROOT / f"fold_{fold['fold']}.json").write_text(
        json.dumps(fold, indent=2), encoding="utf-8"
    )

pd.DataFrame(
    {
        "partition": ["development", "held-out internal evaluation"],
        "patients": [len(dev_ids), len(test_ids)],
    }
)
